# 03 — Graph construction

Builds three edge types for BulkCell-GNN:

| Edge type | Nodes | Method | Purpose |
|---|---|---|---|
| B–B | bulk ↔ bulk | cosine similarity (z-scored expression) | sample-level transcriptional similarity |
| C–C | cell ↔ cell | kNN on PCA (k=15) | cell neighbourhood graph |
| B–C | bulk ↔ cell | top-K cosine on shared genes | cross-modal bipartite edges |

**Key fix in this notebook:** GSE39582 bulk genes are Affymetrix probe IDs (`1007_s_at`).
GSE132465 cell genes are HGNC symbols (`CD3D`). Zero overlap without mapping.
This notebook maps probe IDs → gene symbols using GPL570 platform annotation
before computing any intersection or graph.

**Outputs saved to `DATA_DIR`:**
`edge_BB.pt`, `edge_CC.pt`, `edge_BC.pt`, `edge_BC_weights.pt`,
`bulk_shared.npy`, `cell_shared.npy`, `bulk_expr_sym.npy`, `bulk_genes_sym.npy`, `graph_meta.json`


## 1. Setup — paths

In [9]:
USE_DRIVE = False
from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/BulkCellGNN_data')
else:
    DATA_DIR = Path.cwd() / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR.resolve())


DATA_DIR = C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data


## 2. Install dependencies
> `GEOparse` needed here for GPL570 platform annotation (probe→symbol mapping).

In [10]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'numpy', 'scipy', 'scikit-learn', 'GEOparse'])


0

## 3. Install PyTorch Geometric
> Installs CUDA-matched wheels on Colab A100. Falls back to base package locally (CPU). Safe to re-run.

In [11]:
import sys, subprocess, torch
TORCH = torch.__version__.split('+')[0]
CUDA  = f"cu{str(torch.version.cuda).replace('.', '')}" if torch.version.cuda else 'cpu'
if CUDA != 'cpu':
    wheel_url = f'https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html'
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
            'pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster',
            'torch_spline_conv', '-f', wheel_url])
    except Exception as e:
        print('CUDA wheel install failed, continuing with base package:', e)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'])
print('PyG install done.')


PyG install done.


## 4. Imports and seeds

In [12]:
import os, random, sys, json
from pathlib import Path

SEED = 42
random.seed(SEED)
os.environ.setdefault('PYTHONHASHSEED', str(SEED))
import numpy as np
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import pandas as pd
import GEOparse

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import bulkcell_gnn as bcg

print('Imports OK | device:', 'cuda' if torch.cuda.is_available() else 'cpu')


Imports OK | device: cpu


## 5. Load raw arrays from notebooks 01 and 02
> `bulk_genes` will be Affymetrix probe IDs — confirmed zero overlap with `cell_genes` (HGNC symbols).
> The next cell resolves this via GPL570 platform annotation.

In [13]:
bulk_expr_raw  = np.load(DATA_DIR / 'bulk_expr.npy')
bulk_genes_raw = np.load(DATA_DIR / 'bulk_genes.npy', allow_pickle=True)
cell_expr      = np.load(DATA_DIR / 'cell_expr.npy')
cell_genes     = np.load(DATA_DIR / 'cell_genes.npy', allow_pickle=True)

print(f'Bulk expression : {bulk_expr_raw.shape}  (samples x probes)')
print(f'Cell expression : {cell_expr.shape}  (cells x HVGs)')
print(f'Bulk gene sample: {list(bulk_genes_raw[:3])}')
print(f'Cell gene sample: {list(cell_genes[:3])}')
print(f'Raw overlap     : {len(set(bulk_genes_raw) & set(cell_genes))} shared identifiers')


Bulk expression : (536, 54675)  (samples x probes)
Cell expression : (5226, 19545)  (cells x HVGs)
Bulk gene sample: ['1007_s_at', '1053_at', '117_at']
Cell gene sample: ['A1BG', 'A1BG-AS1', 'A1CF']
Raw overlap     : 0 shared identifiers


## 6. Map Affymetrix probe IDs → HGNC gene symbols (GPL570)

> **Why:** GSE39582 is Affymetrix HG-U133 Plus 2.0 (GPL570). Gene IDs are probe IDs like `1007_s_at`.
> scRNA-seq cell genes are HGNC symbols like `CD3D`. Zero intersection without this mapping.

> GPL570 annotation is cached after the first download — subsequent runs are instant.

> Multi-probe genes (multiple probes → same symbol) are collapsed by **mean** across probes,
> which is standard practice for Affymetrix microarray data going into ML pipelines.

In [14]:
GEO_CACHE = DATA_DIR / 'geo_cache'
GEO_CACHE.mkdir(parents=True, exist_ok=True)

print('Loading GPL570 platform annotation (cached after first run)...')
gpl  = GEOparse.get_GEO('GPL570', destdir=str(GEO_CACHE))
plat = gpl.table.copy()
print(f'Platform table: {plat.shape}')

# Find the gene symbol column
sym_col = None
for c in ('Gene Symbol', 'GENE_SYMBOL', 'gene_assignment', 'Gene_Symbol'):
    if c in plat.columns:
        sym_col = c
        break
if sym_col is None:
    raise RuntimeError(f'Cannot find gene symbol column. Available: {list(plat.columns[:15])}')
print(f'Gene symbol column: "{sym_col}"')
print('Sample rows:')
id_col = 'ID' if 'ID' in plat.columns else plat.columns[0]
print(plat[[id_col, sym_col]].head(4).to_string())


25-Apr-2026 00:09:32 DEBUG utils - Directory C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data\geo_cache already exists. Skipping.
25-Apr-2026 00:09:32 INFO GEOparse - Downloading http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL570&form=text&view=full to C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data\geo_cache\GPL570.txt


Loading GPL570 platform annotation (cached after first run)...


25-Apr-2026 00:09:34 DEBUG downloader - Total size: 0
25-Apr-2026 00:09:34 DEBUG downloader - md5: None
81.6MB [00:02, 30.7MB/s]
25-Apr-2026 00:09:37 DEBUG downloader - Moving C:\Users\krith\AppData\Local\Temp\tmpaqqb0i01 to C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data\geo_cache\GPL570.txt
25-Apr-2026 00:09:37 DEBUG downloader - Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL570&form=text&view=full
25-Apr-2026 00:09:37 INFO GEOparse - Parsing C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data\geo_cache\GPL570.txt: 
25-Apr-2026 00:09:37 DEBUG GEOparse - PLATFORM: GPL570


Platform table: (54675, 16)
Gene symbol column: "Gene Symbol"
Sample rows:
          ID       Gene Symbol
0  1007_s_at  DDR1 /// MIR4640
1    1053_at              RFC2
2     117_at             HSPA6
3     121_at              PAX8


C:\Users\krith\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


In [15]:
# Build probe → first gene symbol mapping
# Affymetrix format: 'SYMBOL1 /// SYMBOL2 /// ...' — take the first symbol only

id_col = 'ID' if 'ID' in plat.columns else plat.columns[0]

probe_to_symbol = {}
for _, row in plat.iterrows():
    probe  = str(row[id_col]).strip()
    symbol = str(row.get(sym_col, '')).strip()
    if not symbol or symbol in ('---', 'nan', '', 'None'):
        continue
    first_sym = symbol.split('///')[0].strip()
    if first_sym:
        probe_to_symbol[probe] = first_sym

print(f'Probes with symbol mapping : {len(probe_to_symbol):,} / {len(plat):,}')
print(f'Probes without mapping     : {len(plat) - len(probe_to_symbol):,} (dropped)')
print('Sample mappings:', dict(list(probe_to_symbol.items())[:5]))


Probes with symbol mapping : 45,782 / 54,675
Probes without mapping     : 8,893 (dropped)
Sample mappings: {'1007_s_at': 'DDR1', '1053_at': 'RFC2', '117_at': 'HSPA6', '121_at': 'PAX8', '1255_g_at': 'GUCA1A'}


## 7. Remap bulk expression to gene symbols and collapse multi-probe genes
> After mapping, multiple probes may point to the same gene symbol.
> These are collapsed by taking their mean — standard microarray practice.
> Result: one row per unique gene symbol, directly comparable to scRNA-seq gene names.

In [16]:
# Map probe IDs to symbols
mapped_symbols = [probe_to_symbol.get(str(p), None) for p in bulk_genes_raw]
n_mapped = sum(1 for s in mapped_symbols if s is not None)
print(f'Probes mapped to a symbol: {n_mapped:,} / {len(bulk_genes_raw):,}')

# Build DataFrame: samples x probes, columns = gene symbols
df_bulk = pd.DataFrame(bulk_expr_raw, columns=mapped_symbols)

# Drop unmapped probes (None columns)
df_bulk = df_bulk.loc[:, df_bulk.columns.notna()]
df_bulk = df_bulk.loc[:, df_bulk.columns != 'None']
df_bulk = df_bulk.loc[:, df_bulk.columns.astype(str) != 'nan']
print(f'After dropping unmapped: {df_bulk.shape}  (samples x probes-with-symbol)')

# Collapse multi-probe genes → mean per gene symbol
# Transpose → group by gene → mean → transpose back
df_bulk_sym = df_bulk.T.groupby(level=0).mean().T
bulk_expr_sym  = df_bulk_sym.values.astype(np.float32)   # (536, n_unique_genes)
bulk_genes_sym = np.array(df_bulk_sym.columns.tolist())
del df_bulk, df_bulk_sym

print(f'After collapsing multi-probe genes:')
print(f'  bulk_expr_sym  : {bulk_expr_sym.shape}  (samples x unique gene symbols)')
print(f'  bulk_genes_sym : {len(bulk_genes_sym):,} unique symbols')
print(f'  Sample symbols : {list(bulk_genes_sym[:5])}')

# Save symbol-remapped bulk arrays for use in notebook 04
np.save(DATA_DIR / 'bulk_expr_sym.npy',  bulk_expr_sym)
np.save(DATA_DIR / 'bulk_genes_sym.npy', bulk_genes_sym)
print('Saved bulk_expr_sym.npy and bulk_genes_sym.npy')


Probes mapped to a symbol: 45,782 / 54,675
After dropping unmapped: (536, 45782)  (samples x probes-with-symbol)
After collapsing multi-probe genes:
  bulk_expr_sym  : (536, 22880)  (samples x unique gene symbols)
  bulk_genes_sym : 22,880 unique symbols
  Sample symbols : [np.str_('A1BG'), np.str_('A1BG-AS1'), np.str_('A1CF'), np.str_('A2M'), np.str_('A2M-AS1')]
Saved bulk_expr_sym.npy and bulk_genes_sym.npy


## 8. Intersect gene symbols — build shared gene space
> This is the gene set used for all cross-modal operations:
> B–B edge construction, B–C cross-modal edges, and input to the model encoders.
> Target: at least 500 shared genes. Expect 2,000–8,000 depending on HVG selection in notebook 02.

In [17]:
bg = {str(g): i for i, g in enumerate(bulk_genes_sym)}
cg = {str(g): i for i, g in enumerate(cell_genes)}

shared = sorted(set(bg.keys()) & set(cg.keys()))
print(f'Shared gene symbols: {len(shared):,}')

if len(shared) < 100:
    raise RuntimeError(
        f'Only {len(shared)} shared genes — something is wrong with the mapping. '
        'Check bulk_genes_sym[:10] and cell_genes[:10] for format differences.'
    )
elif len(shared) < 500:
    print('WARNING: fewer than 500 shared genes. Results may be noisy. '
          'Consider broadening HVG selection in notebook 02.')
else:
    print('Shared gene count is good — proceeding.')

print(f'Sample shared genes: {shared[:8]}')

idx_b = [bg[g] for g in shared]
idx_c = [cg[g] for g in shared]

bulk_shared = bulk_expr_sym[:, idx_b].astype(np.float32)
cell_shared = cell_expr[:,    idx_c].astype(np.float32)

print(f'bulk_shared : {bulk_shared.shape}  (536 samples x {len(shared)} shared genes)')
print(f'cell_shared : {cell_shared.shape}  (5226 cells  x {len(shared)} shared genes)')
assert bulk_shared.shape[1] == cell_shared.shape[1]
assert bulk_shared.shape[1] > 0
print('Shape assertion passed.')


Shared gene symbols: 15,873
Shared gene count is good — proceeding.
Sample shared genes: ['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1', 'A4GALT', 'AAAS', 'AACS']
bulk_shared : (536, 15873)  (536 samples x 15873 shared genes)
cell_shared : (5226, 15873)  (5226 cells  x 15873 shared genes)
Shape assertion passed.


## 9. Build B–B intra-bulk edges
> Cosine similarity on z-scored bulk expression (shared gene space).
> Threshold auto-tuned to keep mean undirected degree between 5 and 30.
> Too sparse (< 5): samples are not connected enough for GAT to aggregate meaningfully.
> Too dense (> 30): graph becomes nearly complete, losing specificity.

In [18]:
print('bulk_shared passed to build_bulk_graph:', bulk_shared.shape)

threshold = 0.70   # starting point — lower than before since z-scored data may have lower cosine sim
for _ in range(15):
    edge_BB, sim = bcg.build_bulk_graph(bulk_shared, gsva_scores=None, threshold=threshold)
    adj      = (sim >= threshold).float()
    N        = adj.shape[0]
    adj_sum  = float(adj.sum().item())
    mean_deg = adj_sum / (2 * N)
    print(f'threshold={threshold:.3f}  mean_deg={mean_deg:.2f}  |E_BB|={edge_BB.shape[1]}')
    if 5 <= mean_deg <= 30:
        break
    threshold = threshold - 0.03 if mean_deg < 5 else threshold + 0.03
    threshold = float(max(0.30, min(0.99, threshold)))

if not (5 <= mean_deg <= 30):
    print('WARNING: degree outside target band — check bulk_shared values.')
else:
    print(f'B-B graph OK: threshold={threshold:.3f}, mean_deg={mean_deg:.2f}')

torch.save(edge_BB, DATA_DIR / 'edge_BB.pt')
print('Saved edge_BB.pt')


bulk_shared passed to build_bulk_graph: (536, 15873)
threshold=0.700  mean_deg=0.19  |E_BB|=204
threshold=0.670  mean_deg=0.24  |E_BB|=252
threshold=0.640  mean_deg=0.27  |E_BB|=288
threshold=0.610  mean_deg=0.31  |E_BB|=328
threshold=0.580  mean_deg=0.35  |E_BB|=374
threshold=0.550  mean_deg=0.42  |E_BB|=450
threshold=0.520  mean_deg=0.53  |E_BB|=566
threshold=0.490  mean_deg=0.69  |E_BB|=740
threshold=0.460  mean_deg=0.90  |E_BB|=968
threshold=0.430  mean_deg=1.29  |E_BB|=1382
threshold=0.400  mean_deg=1.92  |E_BB|=2054
threshold=0.370  mean_deg=2.76  |E_BB|=2954
threshold=0.340  mean_deg=4.04  |E_BB|=4332
threshold=0.310  mean_deg=5.78  |E_BB|=6192
B-B graph OK: threshold=0.310, mean_deg=5.78
Saved edge_BB.pt


## 10. Build C–C intra-cell edges
> Standard scGNN-style kNN graph (k=15) on PCA-reduced cell expression.
> Symmetrised (A = max(A, Aᵀ)) so the graph is undirected.
> Uses shared gene space (same as B–C edges) for consistency.

In [19]:
from sklearn.decomposition import PCA
from sklearn.neighbors import kneighbors_graph

print(f'Running PCA on cell_shared {cell_shared.shape}...')
n_components = min(50, cell_shared.shape[1] - 1, cell_shared.shape[0] - 1)
pca = PCA(n_components=n_components, random_state=SEED)
Z   = pca.fit_transform(cell_shared)
print(f'PCA done: {Z.shape}  ({pca.explained_variance_ratio_.sum()*100:.1f}% variance explained)')

A = kneighbors_graph(Z, n_neighbors=15, mode='connectivity', include_self=False)
A = A.maximum(A.T)   # symmetrise → undirected

try:
    from torch_geometric.utils import from_scipy_sparse_matrix
except ImportError:
    from torch_geometric.utils import from_scipy_sparse_array as from_scipy_sparse_matrix

edge_index_CC, _ = from_scipy_sparse_matrix(A)
torch.save(edge_index_CC, DATA_DIR / 'edge_CC.pt')
print(f'edge_CC saved: {edge_index_CC.shape}  '
      f'(~{edge_index_CC.shape[1]//2:,} undirected edges for {cell_shared.shape[0]:,} cells)')


Running PCA on cell_shared (5226, 15873)...
PCA done: (5226, 50)  (99.8% variance explained)
edge_CC saved: torch.Size([2, 114520])  (~57,260 undirected edges for 5,226 cells)


## 11. Build B–C cross-modal edges (bipartite)
> **This is the architectural core of BulkCell-GNN.**
> For each bulk sample, find the top-K=50 most transcriptionally similar cells
> by cosine similarity on the shared gene space.
> Result: a sparse bipartite graph connecting 536 bulk samples to 5,226 cells.
> Total cross-modal edges: ~536 × 50 = ~26,800 directed edges.

In [20]:
print(f'Building B-C cross-modal edges...')
print(f'  Bulk samples : {bulk_shared.shape[0]}')
print(f'  Cells        : {cell_shared.shape[0]}')
print(f'  Shared genes : {bulk_shared.shape[1]}')
print(f'  Top-K        : 50 cells per bulk sample')

edge_BC, w_bc = bcg.build_cross_modal_graph(
    bulk_shared, cell_shared, top_k=50, batch_size=50
)

torch.save(edge_BC,  DATA_DIR / 'edge_BC.pt')
torch.save(w_bc,     DATA_DIR / 'edge_BC_weights.pt')
print(f'edge_BC saved : {edge_BC.shape}  (row=bulk_idx, col=cell_idx)')
print(f'B-C weights   : {w_bc.shape}  (cosine similarity scores)')
print(f'Weight range  : {w_bc.min().item():.3f} – {w_bc.max().item():.3f}')


Building B-C cross-modal edges...
  Bulk samples : 536
  Cells        : 5226
  Shared genes : 15873
  Top-K        : 50 cells per bulk sample
edge_BC saved : torch.Size([2, 26800])  (row=bulk_idx, col=cell_idx)
B-C weights   : torch.Size([26800])  (cosine similarity scores)
Weight range  : -0.009 – 0.151


## 12. Save shared matrices and graph metadata
> `bulk_shared.npy` and `cell_shared.npy` are the aligned gene-space matrices used
> for graph construction. They are also referenced by the model for alignment loss computation.

In [21]:
np.save(DATA_DIR / 'bulk_shared.npy', bulk_shared)
np.save(DATA_DIR / 'cell_shared.npy', cell_shared)

meta = {
    'n_bulk'            : int(bulk_shared.shape[0]),
    'n_cell'            : int(cell_shared.shape[0]),
    'n_shared_genes'    : int(bulk_shared.shape[1]),
    'n_cell_input_genes': int(cell_expr.shape[1]),
    'n_bulk_input_genes': int(bulk_expr_raw.shape[1]),
    'n_bulk_sym_genes'  : int(bulk_expr_sym.shape[1]),
    'bb_threshold'      : float(threshold),
    'cc_k_neighbors'    : 15,
    'bc_top_k'          : 50,
    'seed'              : SEED,
}
(DATA_DIR / 'graph_meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
print('graph_meta.json saved:')
print(json.dumps(meta, indent=2))


graph_meta.json saved:
{
  "n_bulk": 536,
  "n_cell": 5226,
  "n_shared_genes": 15873,
  "n_cell_input_genes": 19545,
  "n_bulk_input_genes": 54675,
  "n_bulk_sym_genes": 22880,
  "bb_threshold": 0.3099999999999996,
  "cc_k_neighbors": 15,
  "bc_top_k": 50,
  "seed": 42
}


## 13. Final verification — confirm all files exist
> Run this before moving to notebook 04. All six files must be present.

In [22]:
required = [
    'edge_BB.pt', 'edge_CC.pt', 'edge_BC.pt', 'edge_BC_weights.pt',
    'bulk_shared.npy', 'cell_shared.npy',
    'bulk_expr_sym.npy', 'bulk_genes_sym.npy',
    'graph_meta.json',
]

print('File verification:')
all_ok = True
for fname in required:
    p = DATA_DIR / fname
    if p.exists():
        size = p.stat().st_size
        print(f'  OK  {fname:35s} {size/1e6:7.2f} MB')
    else:
        print(f'  MISSING  {fname}')
        all_ok = False

if all_ok:
    print('\nAll files present. Notebook 03 complete — ready for 04_model.ipynb.')
else:
    print('\nSome files missing — re-run the cells above that produce them.')

# Print graph summary
print('\nGraph summary:')
print(f'  B-B edges  : {torch.load(DATA_DIR / "edge_BB.pt").shape[1]:,}')
print(f'  C-C edges  : {torch.load(DATA_DIR / "edge_CC.pt").shape[1]:,}')
print(f'  B-C edges  : {torch.load(DATA_DIR / "edge_BC.pt").shape[1]:,}')
print(f'  Shared genes: {bulk_shared.shape[1]:,}')


File verification:
  OK  edge_BB.pt                             0.10 MB
  OK  edge_CC.pt                             1.83 MB
  OK  edge_BC.pt                             0.43 MB
  OK  edge_BC_weights.pt                     0.11 MB
  OK  bulk_shared.npy                       34.03 MB
  OK  cell_shared.npy                      331.81 MB
  OK  bulk_expr_sym.npy                     49.05 MB
  OK  bulk_genes_sym.npy                     2.75 MB
  OK  graph_meta.json                        0.00 MB

All files present. Notebook 03 complete — ready for 04_model.ipynb.

Graph summary:
  B-B edges  : 6,192
  C-C edges  : 114,520
  B-C edges  : 26,800
  Shared genes: 15,873


## Notebook 03 — Summary

**Root issue resolved:** GSE39582 bulk genes were Affymetrix probe IDs (`1007_s_at`).
GSE132465 cell genes were HGNC symbols (`CD3D`). Raw intersection = 0.
Fix: GPL570 platform annotation used to map probe IDs → gene symbols,
multi-probe genes collapsed by mean. Intersection computed on gene symbols.

**Three graphs built:**

| Graph | Shape | Biological meaning |
|---|---|---|
| B–B | `(2, E_BB)` | Bulk samples connected by transcriptional similarity |
| C–C | `(2, E_CC)` | Cell kNN graph — local cellular neighbourhood |
| B–C | `(2, E_BC)` | Bipartite bulk↔cell links — the cross-modal bridge |

**Key design decision:** B–B edges built on z-scored expression in shared gene space
(not GSVA scores as originally proposed). This is documented as a working prototype
decision; GSVA-based edges are designated future work.

**Next:** Run `04_model.ipynb`. Load `bulk_expr_sym.npy` instead of `bulk_expr.npy`
so the model operates in the same gene symbol space as the cell data.
